# Homework: Monitoring

In module 5 we learned how to monitor our RAG system: capture metrics
from each LLM call, store them in a database, and visualize them on a
dashboard.

In the module we built all of this by hand - a custom dataclass for
the metrics, PostgreSQL for storage, Streamlit and Grafana for
dashboards.

In this homework, we will explore an alternative: [OpenTelemetry](https://opentelemetry.io/) (OTel).
This is the industry standard for code instrumentation. Every monitoring
framework we mentioned is built
on top of it - like Logfire, Langfuse, Arize Phoenix and others.

In this homework we will use OTel directly. We will instrument our
RAG with traces, capture metrics as span attributes, persist the
spans to SQLite, and build a dashboard from the trace data.

We keep using the same course-lessons RAG from homework 1. The
knowledge base is the 72 lesson pages pulled from GitHub, indexed
with minsearch.

> It's possible your answers won't match exactly. If so, select the closest one.

Detailed instructions in [LLM Zoomcamp 2026 Homework 5 instructions](https://github.com/hweecat/llm-zoomcamp/blob/main/cohorts/2026/05-monitoring/homework.md)

In [1]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

In [2]:
from rag_helper import RAGBase

class RAGTraced(RAGBase):
    def rag(self, query: str) -> str:
        with tracer.start_as_current_span("rag") as span:
            return super().rag(query)
    
    def search(self, query: str) -> str:
        with tracer.start_as_current_span("search") as span:
            return super().search(query)
    
    def llm(self, query: str) -> str:
        with tracer.start_as_current_span("llm") as span:
            response = super().llm(query)
            return response

In [3]:
from openai import OpenAI

from gitsource import GithubRepositoryDataReader
from minsearch import Index

COMMIT = "8c1834d"

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id=COMMIT,
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

index = Index(text_fields=["content"], keyword_fields=["filename"])
index.fit(documents)

client = OpenAI()
rag = RAGTraced(index=index, llm_client=client)

## Q1. First trace

Wrap the `rag()` method so each call produces a span. The simplest way
is to create a `RAGTraced` subclass of `RAGBase` that wraps `rag()`,
`search()`, and `llm()` each in their own span.

Run this query:

> How does the agentic loop keep calling the model until it stops?

The console exporter prints every finished span as a dictionary.
Count the spans in the console output - each one is a separate
`ReadableSpan` entry. How many spans does the trace produce?

* 1
* 3
* 5
* 7

### Answer
3

In [4]:
query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

{
    "name": "search",
    "context": {
        "trace_id": "0x4c33e53725a9c93b9c22dcba2640e2c1",
        "span_id": "0x0fb51fdf6a24e475",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xccf949589f1361fe",
    "start_time": "2026-07-20T08:59:10.944571Z",
    "end_time": "2026-07-20T08:59:10.947039Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "c89b216d-3a79-4324-87f4-b30b736d5f6f",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0x4c33e53725a9c93b9c22dcba2640e2c1",
        "span_id": "0x15bf535274b4f595",
        "trace_state": "[]"
    },
    "kind": "SpanKind

## Q2. Capturing metrics as span attributes

Spans are not just timing markers - you can attach any information you
want to them with `set_attribute`. We already use spans to record how
long each step takes. Now we'll add the metrics we care about: tokens
and cost.

Read the token usage from the LLM response (the `llm()` method in the
starter already returns the raw response object) and set them as
attributes on the `llm` span:

```python
span.set_attribute("input_tokens", usage.input_tokens)
span.set_attribute("output_tokens", usage.output_tokens)
```

And since we know both input and output tokens, we can also compute
the cost using the code from the previous modules.

Now re-run the query. How many input tokens do we see?

* 700
* 7000
* 70000
* 700000

### Answer
7000

In [ ]:
class RAGTraced(RAGBase):
    def rag(self, query: str) -> str:
        with tracer.start_as_current_span("rag") as span:
            return super().rag(query)
    
    def search(self, query: str) -> str:
        with tracer.start_as_current_span("search") as span:
            return super().search(query)
    
    def llm(self, query: str) -> str:
        with tracer.start_as_current_span("llm") as span:
            response = super().llm(query)

            # for Q2
            usage = response.usage
            span.set_attribute("input_tokens", usage.input_tokens)
            span.set_attribute("output_tokens", usage.output_tokens)
            return response

In [6]:
query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

{
    "name": "search",
    "context": {
        "trace_id": "0xca398e64716076377ae2dddb00122502",
        "span_id": "0xcba03f0b42e2bd2a",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x1a8729985fd13144",
    "start_time": "2026-07-20T08:59:16.548674Z",
    "end_time": "2026-07-20T08:59:16.553876Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "c89b216d-3a79-4324-87f4-b30b736d5f6f",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0xca398e64716076377ae2dddb00122502",
        "span_id": "0x0fd38af6f4743f73",
        "trace_state": "[]"
    },
    "kind": "SpanKind

## Q3. Span timing

Each span automatically records its duration. Look at the console output
from Q1 and find the durations for the `search` span and the `llm` span.

For a typical query, roughly how long does the LLM call take?

* Under 100ms
* 100-500ms
* 500-2000ms
* Over 2000ms

> The first call can be slower (cold start). Pick the range you see
> most often.

### Answer
500-2000ms

In [7]:
query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

{
    "name": "search",
    "context": {
        "trace_id": "0xedba16a05b06c89729be8b3ba5897225",
        "span_id": "0x833b55834e6a6556",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x1f96bfa1f4e99ae1",
    "start_time": "2026-07-20T08:59:48.976968Z",
    "end_time": "2026-07-20T08:59:48.982038Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "c89b216d-3a79-4324-87f4-b30b736d5f6f",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0xedba16a05b06c89729be8b3ba5897225",
        "span_id": "0x7f20eae2e4558810",
        "trace_state": "[]"
    },
    "kind": "SpanKind

## Q4. Saving traces to SQLite

Right now the spans are printed to the terminal and then gone. We don't
save them.

We want to persist them so we can query them later.

In this homework, we'll use SQLite - it's a more lightweight option than
Postgres, so we don't need to set up any docker containers in this homework.

Our instrumentation is already done, we don't need to change anything there.
But we need to create a custom exporter. Instead of printing the spans,
it will save them to the database.

OTel calls the exporter through the same span processor we already use,
we just swap the destination.

Now we will create a custom exporter that saves each finished span to a
SQLite database. The exporter extends `SpanExporter`. It has the following methods:

- `export` method that receives a list of `ReadableSpan` objects
- `shutdown` and `force_flush` methods

Let's implement it:

```python
import sqlite3
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult


class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True
```

Replace the console exporter with this new exporter:

```python
provider.add_span_processor(
    SimpleSpanProcessor(SQLiteSpanExporter("traces.db"))
)
```

Re-run the query from Q1. Which span names appear in the `spans` table?

* Only `rag`
* `rag` and `llm`
* `rag`, `search`, and `llm`
* `search`, `llm`, and `judge`

### Answer
* `rag`, `search`, and `llm`

Run starter.py file that has been modified to add tracers from tracer.py

In [9]:
import sqlite3

conn = sqlite3.connect("traces.db")

conn.execute("SELECT * FROM spans").fetchall()

[('search', 1784538023349884250, 1784538023352211050, None, None, None),
 ('llm', 1784538023355580903, 1784538026182875233, 7111, 129, None),
 ('rag', 1784538023349787589, 1784538026189695485, None, None, None)]

## Q5. Querying trace data

The traces are now in SQLite. Run one more query through the traced
RAG, then query the database.

The `rag` span wraps everything, so its duration includes both
`search` and `llm`. To see where time actually goes, exclude the
`rag` span and compare the children.

Using SQL (or pandas), compute the total duration for each span name
excluding `rag`. Which span type takes the most total time?

* `search`
* `llm`
* They're all about the same

### Answer
- `llm`

In [11]:
conn.execute("""
    SELECT
    name,
    end_time - start_time AS duration
    FROM spans
""").fetchall()

[('search', 2326800),
 ('llm', 2827294330),
 ('rag', 2839907896),
 ('search', 1725819),
 ('llm', 2118544913),
 ('rag', 2129325008)]

In [40]:
conn.close()

## Q6. Token stability across runs

Load the SQLite data with pandas. One thing a dashboard can tell you
is how stable your system is. If the same query always produces the
same number of input tokens, the context your RAG retrieves is
consistent. If it varies a lot, something in the search may be
unstable.

Run the same query from Q1 three more times (so you have 4 RAG calls
total in the database). Then compute the input tokens for each `llm`
span.

How much do the input tokens vary across these 4 runs?

* They're identical
* Within 10% of each other
* Within 50% of each other
* They vary more than 50%

### Answer
* They're identical

In [12]:
import pandas as pd

with sqlite3.connect("traces.db") as conn:
    spans_df = pd.read_sql_query("SELECT * FROM spans", conn)
spans_df

,name,start_time,end_time,input_tokens,output_tokens,cost
0,search,1784538023349884250,1784538023352211050,NaN,NaN,None
1,llm,1784538023355580903,1784538026182875233,7111.0,129.0,None
2,rag,1784538023349787589,1784538026189695485,NaN,NaN,None
3,search,1784538092128505200,1784538092130231019,NaN,NaN,None
4,llm,1784538092134708358,1784538094253253271,7111.0,79.0,None
5,rag,1784538092128438556,1784538094257763564,NaN,NaN,None
6,search,1784538123529633225,1784538123531405541,NaN,NaN,None
7,llm,1784538123535494786,1784538126204689493,7111.0,89.0,None
8,rag,1784538123529567282,1784538126208483620,NaN,NaN,None
9,search,1784538133356605058,1784538133358962054,NaN,NaN,None


In [13]:
spans_df[spans_df["name"] == "llm"].filter(items=["name", "input_tokens"])

,name,input_tokens
1,llm,7111.0
4,llm,7111.0
7,llm,7111.0
10,llm,7111.0
